# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/protipa_exams_dataset` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [49]:
import json
import logging
import requests
import os
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [50]:
load_dotenv()

# API Config
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("LITELLM_HOST")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-07 16:25:58 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


## 2. Load and Prepare Dataset

In [59]:
logger.info("Loading dataset...")
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")

def filter_dataset(dataset):
    filtered = []
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    for item in dataset:
        multimodal_status = item.get('multimodality')
        if (item['subject'] in subjects and 
            item['exercise_type'] == 'Multiple Choice' and 
            ',' not in str(item['answer_index']) and multimodal_status == 'no'):
            filtered.append(item)
    return filtered

all_filtered = filter_dataset(original_dataset)
logger.info(f"Filtered dataset size: {len(all_filtered)}")

# Sample for pilot test
pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# Save temporary JSON for lm_eval ingestion
json_path = (results_dir / "pilot_data_test.json").resolve()
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(pilot_100, f, ensure_ascii=False, indent=4)

logger.info(f"Pilot data (100 samples) saved to: {json_path}")

2026-01-07 16:45:06 - INFO - Loading dataset...
2026-01-07 16:45:08 - INFO - Filtered dataset size: 192
2026-01-07 16:45:08 - INFO - Pilot data (100 samples) saved to: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\tmp\pilot_data_test.json


## 3. Define Evaluation Task Template

In [65]:
task_config = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "num_fewshot": 3,      # Added few-shot parameter
    "dataset_kwargs": {
        "data_files": str(json_path)  # FIXED: Convert Path object to string
    },
    "test_split": "train",
    "output_type": "generate_until",
    "doc_to_text": (
        "{% if input %}{{input}}\n{% endif %}"
        "Ερώτηση: {{question}}\n"
        "Επιλογές:\n"
        "{% for choice in choices %}"
        "{{loop.index0}}. {{choice}}\n"
        "{% endfor %}\n"
        "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
        "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής.\n"
        "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
        "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
        "❌ ΛΑΘΟΣ: \"Ας υποθέσουμε ότι το κλάσμα είναι 1/12, άρα η απάντηση είναι 2.\"\n"
        "❌ ΛΑΘΟΣ: \"(2)\"\n"
        "❌ ΛΑΘΟΣ: \"2.\"\n"
        "✅ ΣΩΣΤΟ: 2\n\n"
        "Απάντηση: "
    ),
    "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
    "generation_kwargs": {
        "until": ["\n"],
        "max_gen_toks": 50,
        "do_sample": False,
        "temperature": 0.0 
    },
    "filter_list": [
        {
            "name": "strict-match",
            "filter": [
                {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                {"function": "take_first"}
            ]
        }
    ],
    "metric_list": [
        {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
    ]
}

# Workaround for Task Config loading
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)
with open(task_dir / "greek_protipa.yaml", "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_dict = {"greek_protipa_exams": custom_task}
logger.info("Evaluation task defined successfully.")

2026-01-07 17:17:27 - WARNING - [Task: greek_protipa_exams] num_fewshot > 0 but fewshot_split is None. using preconfigured rule.
2026-01-07 17:17:27 - WARNING - [Task: greek_protipa_exams] num_fewshot > 0 but fewshot_split is None. using preconfigured rule.
2026-01-07 17:17:28 - INFO - Evaluation task defined successfully.


## 4. Run Evaluation

In [66]:
comparison_results = {}
all_samples = {}

EVAL_LIMIT = 100  # Adjust this to run more/less samples

for model_name in models_to_test:
    logger.info(f"Starting evaluation for model: {model_name}")
    try:
        # Construct endpoint URL
        chat_api_url = api_base
        if not chat_api_url.endswith("/chat/completions"):
            chat_api_url = chat_api_url.rstrip("/") + "/chat/completions"

        model = OpenAIChatCompletion(
            model=model_name,
            base_url=chat_api_url,
            num_fewshot=0,
            eos_string="<|end_of_text|>",
            max_retries=10,
            num_concurrent=1
        )

        results = lm_eval.evaluate(
            lm=model,
            task_dict=task_dict,
            limit=EVAL_LIMIT,
            apply_chat_template=True
        )
        
        # 1. Store Summary Metrics
        scores = results['results']['greek_protipa_exams']
        comparison_results[model_name] = scores
        
        # 2. Collect Individual Samples for the table
        if 'samples' in results and 'greek_protipa_exams' in results['samples']:
            samples = results['samples']['greek_protipa_exams']
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('school_level', 'N/A'),
                        "Ground Truth": str(doc.get('answer_index', 'N/A')),
                        "Choices": " | ".join(doc.get('choices', [])),
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
        logger.info(f"Success! {model_name} Accuracy: {acc:.2%}")
        
        logger.info("Waiting 2 seconds before next model to avoid rate-limits...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"Error evaluating {model_name}: {e}")
        logger.error(traceback.format_exc())
        time.sleep(2)

2026-01-07 17:17:31 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-07 17:17:31 - INFO - Using max length 2048 - 1
2026-01-07 17:17:31 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-07 17:17:31 - INFO - Using tokenizer None
2026-01-07 17:17:31 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-07 17:17:31 - INFO - Building contexts for greek_protipa_exams on rank 0...




























100%|██████████| 100/100 [00:00<00:00, 139.04it/s]
2026-01-07 17:17:31 - INFO - Running generate_until requests
2026-01-07 17:17:31 - INFO - Tokenized requests are disabled. Context + generation length is not checked.






















































































































































































































## 5. Results Table

In [69]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():
        row = {
            "ID": idx,
            "Subject": data["Subject"],
            "Year": data["Year"],
            "Level": data["Level"],
            "Question": data["Question"],
            "Choices": data["Choices"],
            "Ground Truth": data["Ground Truth"]
        }
        for m in models_to_test:
            # Add both the extracted prediction AND the raw text from the model
            row[f"{m}_pred"] = data["Model Predictions"].get(m, "N/A")
            # row[f"{m}_raw"] = data["Raw Responses"].get(m, "N/A") 
            
        table_data.append(row)
    
    df_results = pd.DataFrame(table_data)
    
    # Save CSV
    results_file = results_dir / "eval_results_table.csv"
    df_results.to_csv(results_file, index=False, encoding='utf-8-sig')
    logger.info(f"Table saved to: {results_file}")
    
    # This will now display the raw responses in the table below
    display(df_results.head(10)) 

2026-01-07 17:24:45 - INFO - Table saved to: tmp\eval_results_table.csv


,ID,Subject,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,ΜΑΘΗΜΑΤΙΚΑ,2020,ΛΥΚΕΙΟ,Το άθροισμα δύο θετικών αριθμών είναι 5πλάσιο ...,A. $\frac{5}{4}$ | B. $\frac{3}{2}$ | Γ. $\fra...,1,3,2
1,1,ΓΛΩΣΣΑ,2021,ΓΥΜΝΑΣΙΟ,Σύμφωνα με την ερμηνεία της γιαγιάς «στοιχειό ...,α. ο πατέρας γυρνούσε τραυματισμένος κάθε βράδ...,1,1,0
2,2,ΓΛΩΣΣΑ,2018,ΓΥΜΝΑΣΙΟ,Τον πιο σημαντικό ρόλο στην εκπαίδευση των ανθ...,α. η οικογένεια | β. το σχολείο | γ. οι τέχνες...,1,0,1
3,3,ΓΛΩΣΣΑ,2020,ΛΥΚΕΙΟ,**πως η ιστορία μου απαρτιζόταν από δύο ξεχωρι...,βουλητική που λειτουργεί ως επεξήγηση | βουλητ...,3,2,2
4,4,ΜΑΘΗΜΑΤΙΚΑ,2024,ΓΥΜΝΑΣΙΟ,Ποιος αριθμός μεταξύ 20 και 35 λείπει από το μ...,A. 26 | B. 27 | Γ. 28 | Δ. 29,1,2,2
5,5,ΜΑΘΗΜΑΤΙΚΑ,2018,ΓΥΜΝΑΣΙΟ,Αν 3 (όμοιες) μπουκάλες γεμίζουν με 4 γεμάτες ...,Α. 8 | Β. 10 | Γ. 12 | Δ.21 | Ε. 28,1,2,2
6,6,ΓΛΩΣΣΑ,2017,ΓΥΜΝΑΣΙΟ,Με ποια από τις παρακάτω λέξεις θα αντικαθιστο...,Α. αρχή | Β. καταγωγή | Γ. ρίζα | Δ. πηγή,1,1,2
7,7,ΜΑΘΗΜΑΤΙΚΑ,2021,ΓΥΜΝΑΣΙΟ,Από τους παρακάτω αριθμούς ο μικρότερος είναι:,Α. Το 5% του 40 | Β. Το 10% του 15 | Γ. $T\alp...,1,1,2
8,8,ΜΑΘΗΜΑΤΙΚΑ,2017,ΓΥΜΝΑΣΙΟ,Ο Ορφέας αποφάσισε να ασχοληθεί με το τρέξιμο....,Α. 1 λεπτό λιγότερο | Β. 2 λεπτά λιγότερο | Γ....,0,1,2
9,9,ΓΛΩΣΣΑ,2020,ΓΥΜΝΑΣΙΟ,"Ποια από τις παρακάτω προτάσεις είναι σωστή, σ...",Α. Στο Μαντείο των Δελφών δίνονταν ξεκάθαρα κα...,3,3,3


## 6. Performance Summary

In [68]:
if comparison_results:
    summary_data = []
    for model, metrics in comparison_results.items():
        acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
        summary_data.append({"Model": model, "Accuracy": acc})
    
    df_summary = pd.DataFrame(summary_data)
    display(df_summary)

,Model,Accuracy
0,gemma3-27b-it,0.50
1,krikri-dpo-context,0.34


Διαθέσιμα μοντέλα

In [48]:
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        # Φιλτράρουμε μόνο αυτά που μας ενδιαφέρουν
        targets = ['llama', 'mistral', 'gemma', 'meltemi']
        found_models = []
        
        for m in models:
            mid = m['id']
            # Αν το ID περιέχει κάποια από τις λέξεις κλειδιά
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "llama-3.2-1b",
    "mistral-small-24.02",
    "llama-3.1-70b",
    "gemma3-27b-it",
    "llama-3.2-3b",
    "mistral-7b-instruct-v0.2",
    "llama-3.1-8b",
    "llama-3.3-70b",
    "mistral-large-24.02"
]


○ Τα υπόλοιπα μοντέλα (πχ llama-3.1-8b, mistral-7b-instruct-v0.2) σκάνε με Bedrock error όταν τα τρέχω

○ Το accuracy του krikri και του gemma ήταν 29% και 48% αντίστοιχα όταν τα τρέχω με το multimodality == 'yes'

○ Θέτοντας το multimodality == 'no', πέφτει η ακρίβεια και των 2 μοντέλων σε 22% (krikri) και 43% (gemma)

○ Το krikri σε zero-shot setting καταρρέει και "κλειδώνει" ότι η σωστή απάντηση βρίσκεται πάντα στο index 2 σε όλες τις απαντήσεις που δίνει  

○ Αλλάζοντας τα settings του task_config σε "num_fewshot": 3, αυξάνονται τα ποσοστά της ακρίβειας σε 50% για το gemma και 34% για το krikri χωρίς να κλειδώνει στο index 2, δίνοντας κι άλλες θέσεις (0, 1, 3)